# Theory behind persistent homology features (PHF)

## Why PHFs?

Chemical structure is an inherently 3-D property and is essential for relating the shape of a molecule to its function. To solve chemical problems, for example, to develop better drugs, materials and chemicals, to understand biochemistry and thus how life works, to create novel biotechnology, to predict the outcome of a reaction and to plan how to make a specific molecule, we need to use computers. The field of machine learnning and artifical intelligence could be of great use. 

However, how best to input molecular structure into an ML algorithm is not a solved problem. Featurisations have been invented that are 1-D an exclude the molecular structure, 2-D which include the molecular graph (chemical formula) but exclude 3-D structure, and 3-D voxel-based methods, which are very hard to train with as the resulting featurisation is very sparse. 

PHFs aim to solve this problem by encoding the 3-D structure into a 1-D vector that is non-sparse. 

Another problem that chemistry has is the size of the datasets, they are much smaller than those that have worked well with ML. Thus, we must get the most out of every datapoint and we cannot use huge models and large amounts of compute.

So, we must make the problem simpler for the ML algorithm, byu doing pre-processing to pick out what is chemically useful and only give the algorithm that. 

It was thought (and shown to be true) that using topological shape might create a compact, efficient and useful measure of 3-D chemical structure, that only requires small simple algorithms. (It was a surprise that) this worked, and it was found that PHF features offered a 10-100 times speed up on training time, power usagae and reduced dataset size by the same amount. Using just PHFs gave results comparable to the state of the art on benchmark problems, adding in some cheap to calculate chemical featureas (rdkit features) gave results that equalled the best of the state of the art, all with much faster training time and much simpler models (the most complex model we use in a 2-layer (2 time 100 units) feed-forward neural network and no parameter optimisation is required). Analysing which problems work best with just topological features also tells us which problems are most affected by chemical shape.

See: 
Ella Gale, "Shape is (almost) all!: Persistent homology features (PHFs) are an information rich input for efficient molecular machine learning.", forthcoming 2026

## Persistent homology

Persistent homology is the extension of topological shape ideas to point clouds. Roughly, topological shape involves counting the number of connected components and holes in a surface. Persistent homology does this point clouds by putting a ball of known radius around each point and merging them as the balls' radii get bigger, giving us a surface in which we count the holes. This addition of a ball radius to the method means that we can talk about many lengthscales a hole or connected region persists for.

# Methodology

## Point clouds

First, we turn atom positions to point clouds, this removes atom type, formal bonds, atom size, valence, mass and charge, leaving *only* shape information. The only chemical information left is that which could be inferred from the distances between atoms by the ML algorithm.

![buckyball](pictures/buckyball_pointcloud.png)

## Simplices

![simplices](pictures/simplices.png)


It might seem strange, given that our interest is in curved surfaces, which are the underlying smoothed ;shape' of a molecule, the next step is to use straight and flat objects to quantify this shape. This is because topology does not differentiate between curved and straight lines, and we are only interested in the connections between parts, thus we make the task simple and use simple objects called simplices to quantify this connection.

3-D simplices are very familiar to chemists as they are tetrahedrons, the concept can be extended to any dimensionality of space, however only four concern us here: the point (0-D), the line (1-D), the triangle (2-D) and the tetrahedron (3-D), see figure above. Note that simplices can have parts of any length, they do not need to be equal as drawn above. A *polytope* is a geometric object with 'flat' sides and a simplex is the simplest possible polytope in any given space. For example, a triangle is a flat 2-D object with three sides, and adding more sides makes a less simple polytope, removing one side makes two 1-D lines (i.e. not a polytope). The best way to think of 'flatness' is that the tangents to the $n$-simplex is are parallel in the $n+1$ dimension. i.e. a line in $x$ has parallel tangents all along it in $y$, a triangle in 2-D space $(x,y)$ has parallel tangents all along the face in $z$ and so on, see also Theory2 section flatness.

**Table 1. Example topological surfaces (manifolds) and their Betti numbers**

| Manifold  | Dimension | $\beta_0$ | $\beta_1$ | $\beta_2$ |
|-----------|-----------|-----------|-----------|-----------|
|Circle, $S^1$ | 1-D | 1 | 1 | -|
|Sphere, $S^2$ | 2-D | 1 | 0 |  1 |
| Torus, $T^2$ | 2-D  | 1| 2 |  1|






**Table 2. Terminology**


|Topological invariant | Properties that remain constant under topological deformation.|
|----------------------|---------------------------------------------------------------|
|Homeotopically equivalent  | Same topological shape, one can be continuously deformed into the other, i.e. you can map $X$ to $Y$.|
|Homeomorphically equivalent | Like homotopically equivalent, but you can deform $X$ to $Y$ and $Y$ to $X$.|
|Connected space | Not a union of two disjoint non-empty open sets. |
|$H_k (X)$ | Homology group $k$ of $X$, generates paths around holes of dimension $k$.|
|Betti numbers $\beta_{k}$ |  Rank of $H_k$, the number of cuts that can be made before $X$ is no longer connected.|
|$\beta_0$ | Number of connected components.|
|$\beta_1$ | Number of 1-D `flat' holes, |
|$\beta_2$ | Number of 2-D holes (voids). |


## Building simplical complexes

If a ball of radius $\eta$ is centred on two atoms, now modeled as points, and the radius of that ball is half the distance between them such that the balls touch or overlap, those points are now connected. In chemistry, this generally (but, crucially, not always) corresponds to a formal bond. The next step of the calculation would be to draw a 1-simplex (straight line) between the two 0-simplices (points) that describe the object. For three atoms, we draw a triangle between them if the balls overlap. Simplices are hierarchical, so a triangle consists of three 0-simplices (points), three 1-simplices (lines) and one 2-simplex (triangle). Ditto a tetrahedron four 0-simplices (points), six 1-simplices (lines) and four 2-simplices (triangle) and one 3-simplex (tetrahedron)\footnote{A. Higher order simplices can exist in higher dimensional space. B. You can create cubical complexes from points, lines, squares and cubes, but these are less simple as they have more lower-order components.}. Note that the torus presented in figure~\ref{fig:topol_example} is actually a simplical complex built on a torus using line simplices.

For a threshold $\eta$ the procedure is as follows:

1. add a line (1-simplex) between any two points that are less than $\eta$ from each other (i.e. balls overlap)
2.  add a triangle (2-simplex) between any three points that are less than $\eta$ from each other
3. add a tetrahedron (3-simplex) between any four points that are less than $\eta$ from each other.
\end{enumerate}

The result is a *simplical complex* (a connected object made up of simplices).

We write simplices as the points that define them, so a line between $a$ and $b$ is written as $[a, b]$, and this is actually the set of all points between $a$ and $b$ that satisfy the constraints for `flatness' (see also Theory 2~ section implical_complex).

## Creation of Vietoris-Rips complex

A series of simplical complexes are created at different ball sizes and these are combined into a heirachical complex called a Vietoris-Rips complex. 

## Barcodes and perisistence diagrams

The filtration at which a generator appears is called birth, that at which is dies is called death, and 
$$
\mathrm{persistence} = \mathrm{death} - \mathrm{birth} \:,
$$
thus we can plot the persistence of generators in persistence diagrams.

Persistence diagrams are a compact and easy way to represent the information in a barcode plot (especially when you have many points) as the filtration is changed, i.e. the balls' diameter is increased. Birth refers to the creation of a feature, death when that feature dies (by merging with another feature). Points on the birth-death line would be points created and then immediately destroyed, and many of the points on this line correspond to non-persistent features and are designated as noise.\cite{atienza2016separating} The more persistent a feature, the more important it is considered. Those points that persist over a range of filtrations are high off the line, and we read the diagrams by looking at these to find out what homology group generators are present (i.e. how many clusters, holes and voids).

## Vectorisation of persistence diagrams

Persistence diagrams are easy for humans to interpret but not very useful for inputting into a machine learning algorithm. To do that we use five features derived from the persistence diagram: Persistence entropy, No. of points, Bottleneck distance, Wasserstein amplitude, Persistence image, and each feature is 3-D giving an input vector of 18 numbers long for each input. A description of how these features are calculated is in the Theory 2. 

Molecules are turned to point clouds with no atomic identifiers and converted Vietoris-Rips complexes as in step 1 above. Barcode plots and persistence diagrams encode the persistence of shape features over length scales. Persistence diagrams are vectorized into 1-D vectors of 18 numbers and used as a very small and efficient input to machine learning algorithms.

# Worked example using benzene

To make the concepts in the previous section clear and to investigate what information is preserved by persistent homology featurisation we will now go through the method applied to a very simple molecule: benzene.

**Figure 3: Method**

![Method](pictures/method_1.png)

## Creating Vietoris-Rips complex.

First we make a point cloud of the atoms as shown in figure 3a. We then increase the value of $\eta$ to make a series of complexes and record where homological groups ($H_k$) are born and die. Betti numbers, $\beta_k$, are the number of features of a given homological type. 

We record all the simplical features in complexes $X_n$.

If $\eta$ is less that $^{r_{CH}}/_2$, we only have points:
$$
X_0 = \{C_1, C_2, ... C_6, H_1, H_2, ... H_6 \}
$$
and the Betti numbers are $\beta_0 = 12$, $\beta_1 = 0$, $\beta_2 = 0$, i.e. there are 12 connected components and the complex $X_0$ consists of a set of points. Here we have labelled them with the atoms for ease of understanding but in the method these are equivalent.

If $\eta$ is more than $^{r_{CH}}/_2$ but less than $^{r_{CC}}/_2$, 1-simplices are formed between the C-H atoms: we have 12 points (simplex-0) and 6 lines (simplex-1):
$$
X_1 = \{X_0, [C_1, H_1], ... [C_6, H_6],
$$
as we now have 6 connected components (clusters) (that correspond to the cluster of C and H points in a CH bond), see figure 3c

, thus $\beta_0 = 6$, $\beta_1 = 0$, $\beta_2 = 0$, topologically this is 6 clusters.

If $\eta$ is more than $r_{CC} / 2$ we have 12 points and 12 lines and 12 triangles:
$$
X_2 = \{X_0, X_1, [C_1, C_2], ... [C_6, C_1], 
$$
$$
[H_1, C_1, C_2], [H_1, C_1, C_6], ... 
$$
$$
[H_6, C_6, C_5], [H_6, C_6, C_1]\} \;,
$$
The CCH triangles overlap creating a loop, see figure3 as this is a 1-D surface we now know that we have a 2-D hole in this structure, and the Betti number, $\beta_1 = 1$. So this simple has $\beta_0 = 1$, $\beta_1 = 1$, $\beta_2 = 0$. At this point benzene is homotopically equivalent to a circle ($S^1$), see table 1. This is conceptually slightly odd for a chemist as the benzene ring structure has been built from HCH triangles overlapping rather than the ring of CC bonds. 

Increasing $\eta$, we add in the CCC angles as 2-simplices:
$$
X_3 = \{X_0, X_1, X_2, [C_1, C_2, C_3], ... [C_6, C_1, C_2]\}
$$
see figure~\ref{fig:benzene_simplex}d.
As the topology was already that of a circle, topologically nothing is changed by this addition. We have 12 points, 24 lines and 18 triangles. At $\eta = 2.432$ the hole at the centre switches from a 1-D 'flat' hole to a 2-D void.

When $\eta$ is half the diameter of a benzene molecule, we add connections across the ring, and new triangles:
$$
X_4 = \{X_0, X_1, X_2, X_3, [C_1, C_4], ... [C_3, C_6], [C_1, C_3, C_4], [C_6, C_2, C_3]\}
$$
and the Betti numbers are $\beta_1 = 1$, $\beta_1 = 0$, $\beta_2 = 1$ and the circle dies, 

Eventually, the hydrogens across the ring join:
$$
X_5 = \{X_0, X_1, X_2, X_3, X_4, [H_1, C_1, C_4, H_4], ... [H_6, C_6, C_3, H_3]\}
$$
and everything is connected, and benzene is now homotopically equivalent to a sphere and is a single cluster, see figure 3e, $\beta_0 = 1$, $\beta_1 = 0$, $\beta_2 = 0$ 
%As this join can be thought of as going over the top, the hole(s) that are joined in 3-D. If you think about the hydrogens being slightly bent up and out of the plane it is obvious that a 3-D void would be formed above the plane of the carbons. The rdkit version is only flat to five decimals places (in fact the six C-H bonds did not have the exact same value but this cannot be seen in the persistence plot as they overlap), which can explain the presence of the void. Although, if the benzene is perfectly flat then from the reasoning of the amount the hydrogens are above the plane tending to zero there is an infinitesimal point where the void appears and disappears. This would appear on the birth=death line.)

I expect that this example has also made clear the links between topology and combinatorics. The final result is the Viertoris-Rips complex, which is a set of sets of complexes, as illustrated in figure 4.

From the benzene example, it is tempting to relate the simplices to something a chemist would recognise and name, i.e bonds, angles and torsions, thinking this way is problematic for two reasons. Firstly, angles and torsions are geometric properties, expecting straight lines and measurable angles. Topology does not measure or preserve angles, straight-lines are equivalent to curves, (We're using straight lines and flat shapes for convenience, and they are the simplest way to draw connections, hence simplices.) and there are no atom labels being used, so when we write [C, H, H] it is just connection between three points in space. Secondly, not all the connected parts are something a chemist would recognise and label. For example, the hydrogens above cyclohexane in a boat configuration will be joined by simplices but are five bonds away. Nearby atoms on neighbouring protein chains would also be joined. Counter ions also. Note that, if this topological approach works for a problem we can assume that these longer range properties are required to solve the problem.

**Figure 4. Vectoris-Rips complex for benzene. The complex is hierarchical and compositional; higher values of $\eta$ give complexes composed of the simplices at that ballsize and the simplical complexes found at smaller radii.Only the largest non-zero Betti numbers are shown.**

![VR](pictures/VR_complex.png)



## Creating the barcode plot

Figure 4 shows the final Veirtoris-Rips complex as a heirarchical set of sets of simplices relating to discrete values of $\eta$, however the topological features exist over a continuum of length-scales. We are interested in the persistence of these homological generators at different length-scales, which is what is plotted against a continuous scale in the barcode plot. 

A barcode plot records the birth and death of homological features, see figure 5 (top), over the length-scale (called filter in this field as we are filtering the simplical complexes).  There are 12 points (simplex-0) in benzene, there is always a point feature that continues to infinity (which we do not plot), so there are only 11 points on the barcode plot in figure 5(top). The 12 connected regions are born at 0 Angstroms $\,$ and coloured blue on the diagram. Six of them die at the C-H bond length when they merge with the other six to create 6 clusters.  Note that the atoms are not numbered or differentiated, so there is no difference between them, and the ordering over the barcode plot is merely for convenience.\footnote{It might be interesting to plot the birth and death of these features ordered as distance from a point to give a spatial ordering over the features, but this not pure topology and not investigated in this paper.} Six of these simplex-0 die at 1.08 Angstroms, which is the C-H bond distance, and this corresponds to figure 3 and simplical complex $X_1$. At a filter of 1.4 Angstroms, the C-C bond length, the last remaining clusters die and a new feature, of homology group $H_1$ is born, i.e. the clusters merge to become a 2-D hole with a 1-D surface around it (the carbon ring), this is simplical complex $X_2$, see figure 3. This ring persists for 1.024 Angstroms before it closes (simplical complex $X_4$ and figure 3). At this filter value, benzene is indistinguishable from a single cluster, and any increases in ball size will not change the homology of the complex. This single cluster persists until $\infty$\AA, but this final point is not plotted on the barcode plot, and this explains why there are only 11 features in figure 5 (top). 

**Figure 5. The barcode plot (top) and Betti curve (bottom) for benzene.**

![barcode](pictures/barcode_benzene_replot.png)

![betti curve](pictures/benzene_betti_curve.png)




## Persistence diagram

**Figure 6. Benzene persistence diagram**
![benzene_persistence_diagram](pictures/benzene_persistence.png)

Each topological feature is described by a pair of numbers, $\eta$ and $t$, where $\eta$ is the current ball radius and $t$ is the maximum radius achieved. 

This is converted to a multiplicity of points $(b, d)$ (see Theory2) where $b$ is the radius where the feature is born and $d$ is the radius where is dies. The persistence diagram, despite being called a diagram, is actually the multiset of points as shown for benzene in table 3}. Each topological feature has three numbers associated with it, the radius at which it is born, the radius at which it dies and the Betti number for that feature. There is a final feature that is often excluded from persistence diagrams and that is the feature that starts at the point that the penultimate feature dies (the last row in table 3) and dies at infinity--this is not plotted. (This has a nice metaphorical interpretation which fits with our understanding of the human visual system, at large distance, every shape becomes a point.) For molecules, there will always be $N-1$ original features for this reason (notice that one of the hydrogens are missing from the barcode plot).  

The data in the persistence diagram in table 3 can be plotted on something which is confusingly also called a persistence diagram, and examples of these are shown in figure 6. These persistence diagrams plot the data as multisets of points $(b, d)$ i.e. we plot birth against death with the Betti number (third column) shown as colour. (multisets defined in Theory2). 

If there are several points with the same $(b, d)$ they are plotted on top of each other. In the software, hovering a mouse over the points reveals the multiplicity of that value, one could also annotate the diagram with the number next to it. Obviously, molecules with a high degree of symmetry and those in the perfect ground structure will have persistence diagrams with a high degree of multiplicity. As molecules move, or the structure changes slightly these multiple points move apart, as shown in in figure~\ref{fig:6_C}. 

**Table3. Persistence diagram for benzene**

|Birth | Death | $k$ in $H_k$ |
|------|-------|--------------|
|0.        | 1.08233893 | 0.       |
|0.        |  1.08233905 | 0.      |
|0.        |  1.08233905 |  0.   |
|0.        |  1.08233905 | 0.    |
|0.        |  1.08233905 |  0.    |
|0.        | 1.08233905 | 0.     |
|0.        |  1.39904296 | 0.      |
|0.        |  1.39904296 | 0.    |
|0.        |  1.39904296 |  0.     |
|0.        |  1.39904296 | 0.    |
|0.        |  1.39904296 | 0.     |
|1.39904296 | 2.42321348 |  1.    |
|2.42321348 |  2.79808593 | 2.    |


What is the homology class of a benzene molecule? This is where persistence is useful, we choose to discard features that are not persistent. There is not a formal mathematical rule for doing this. Generally we might approximate it as the highest ranking $H_0$ with the largest persistence, and from the data above, that would suggest that benzene is homotopically equivalent to a circle, i.e. it has one (flat) hole. The connected regions ($H_0$) also persist for a long time, which tells us that this larger hole structure is made of clusters, which makes chemical sense as we do think of benzene as being a (ring) made of points (atoms). In topological data analysis the closer the point is to the line, the more likely it is to be noise. In these chemical structures these `noisy' points seem to encode something interesting about the structure. For example, in benzene, the $H_2$ could be ignored as an artifact of the method as having large spheres close together leaves a spherical type hole in the middle. (I.e. a hole like a octahedral or tetrahedral hole found in crystals, note that these are all homeotopically equivalent.)

It is worthwhile to ponder what the mathematics is telling us about the structure of benzene. As there are 12 $\beta_0$ (remember only 11 are plotted) which persist, benzene can be thought of as having 12 small parts, which is obviously the atoms. As both a $\beta_1$ and $\beta_2$ feature are found, with the $\beta_1$ being more persistent, benzene has a charactor of a loop (the ring structure), which is mostly flat (as $\beta_1$) in character. At a large enough $\eta$, equivalent to `zooming out' benzene is just a single connected region, which we would call a molecule and might well model as just a particle. As persistence diagrams also include information about the length scale, there is some measure of size of these features included in the description.

The process for creating these topological features may seem rather involved, however, it is important to note that this can be done in only four lines of python code using `GandT` (see Appendix \ref{sec:code_example}). The interested reader is referred to Appendix~\ref{sec:maths} for further, more formal mathematical detail.


Homology only counts topological shape, but the filtered complexes used in persistent homology include some aspect of geometry due to the fact that the simplical complexes are built from points in space and their relations. As a result, the two different forms of cyclohexane are demonstrably different, see their persistence diagrams in `Analysis of persistence homology for molecules.ipynb`. Note that graph topology would not differentiate between these two structures as the graph connection between the atoms is the same in both cases.

Persistent homology only looks at shape and loses information about atom types, so information about enantomers is largely lost, for example, the persistence diagram of S- and D-alanine are the same to three decimal places (and the difference at that level is likely due to numerical calculations). 